# AFL Players Data Cleaning and Validation Project

**Objective:** Validate, clean, and merge `afl_players_info_raw.csv` and `afl_players_seasonal_stats_raw.csv` into a single analysis-ready dataset.


## Data Quality Assessment Report

### **afl_players_info_raw.csv**
**Initial Shape:** 2848 rows, 16 columns.

**Duplicates:** 5 duplicate records identified.

**Missing Values:**
- `profile_pic`: 2211 missing values.
- `player_common_names`: 2773 missing values.
- `player_teams`: 94 missing values.

**Invalid Values:** 2 rows have `weight` = 0, which isn't a real player weight.

**Data Types:** Player IDs are integers, `born_date`/`debut_date`/`last_date` are stored as text, remaining columns are mainly strings.

### **afl_players_seasonal_stats_raw.csv**
**Initial Shape:** 25491 rows, 54 columns.

**Duplicates:** 10 duplicate records identified.

**Missing Values:** Widespread missing values across most numerical statistics columns (e.g. goals, kicks, marks) indicating unrecorded stats.

**Invalid Values:** 8 rows have negative `games_played`, which isn't possible.

**Inconsistent Formatting:** `team` names appear in different cases/spacing (e.g. "Carlton Blues" vs "CARLTON BLUES").

**Data Types:** `player_id` has mixed types (strings and integers) with some invalid/non-numeric representations.

In [1]:
import pandas as pd
import numpy as np

# Load Data
info_df = pd.read_csv("afl_players_info_raw.csv")
stats_df = pd.read_csv("afl_players_seasonal_stats_raw.csv", low_memory=False)

print(f"Info Shape: {info_df.shape}")
print(f"Stats Shape: {stats_df.shape}")


FileNotFoundError: [Errno 2] No such file or directory: 'afl_players_info_raw.csv'

## Data Cleaning Log

### 1. Players Info Dataset (`players_info.csv`)
- **Duplicates:** Removed 5 duplicated rows to ensure each player record is unique.
- **Missing Values:**
  - `profile_pic`: Replaced missing URLs with 'No Image'.
  - `player_common_names`: Replaced missing values with 'Unknown'.
  - `player_teams`: Replaced missing values with 'Unknown'.
- **Invalid weight:** Replaced `weight` = 0 with 'Unknown' rather than a fabricated number.
- **Dates:** Converted `born_date`, `debut_date`, `last_date` to proper date format.
- Rationale: Instead of dropping rows and losing valuable player demographics, we impute placeholders for missing/invalid values.

### 2. Seasonal Stats Dataset (`seasonal_stats.csv`)
- **Duplicates:** Removed 10 duplicated rows to prevent double-counting stats.
- **Invalid IDs:** Converted `player_id` to numeric. Coerced non-numeric invalid string characters (e.g. `"ID_43261"`) to NaN and dropped those rows. `player_id` was then cast to standard integer.
- **Negative games_played:** Took the absolute value, since these were clearly sign errors (e.g. -13 games instead of 13).
- **Team names:** Standardised casing/spacing (e.g. "CARLTON BLUES" -> "Carlton Blues").
- **Missing Values:** Replaced missing numerical statistics (like kicks, goals, behinds) with 0.
- Rationale: In sports statistics, a missing value for an action (e.g. goals) typically implies the action did not occur (0 instances), rather than being truly unknown. Casting `player_id` to an integer guarantees a consistent key for merging.

In [2]:
# Clean Players Info
info_df = info_df.drop_duplicates()
info_df['profile_pic'] = info_df['profile_pic'].fillna('No Image')
info_df['player_common_names'] = info_df['player_common_names'].fillna('Unknown')
info_df['player_teams'] = info_df['player_teams'].fillna('Unknown')
info_df['weight'] = info_df['weight'].replace(0, np.nan).fillna('Unknown')

for col in ['born_date', 'debut_date', 'last_date']:
    info_df[col] = pd.to_datetime(info_df[col], errors='coerce')


NameError: name 'info_df' is not defined

In [5]:
# Clean Seasonal Stats
stats_df = stats_df.drop_duplicates()
stats_df['player_id'] = pd.to_numeric(stats_df['player_id'], errors='coerce')
stats_df = stats_df.dropna(subset=['player_id'])
stats_df['player_id'] = stats_df['player_id'].astype(int)

stats_df['games_played'] = stats_df['games_played'].abs()
stats_df['team'] = stats_df['team'].str.strip().str.title()

# Fill missing numerical stats with 0
numeric_cols = stats_df.select_dtypes(include=[np.number]).columns
stats_filled = stats_df[numeric_cols].isna().sum().sum()
stats_df[numeric_cols] = stats_df[numeric_cols].fillna(0)


## Data Merging

We perform a left join from `seasonal_stats` to `players_info` using the `player_id` and `id` keys respectively. This ensures we retain all statistical records while attaching demographic information where available.

In [6]:
# Merge Datasets
merged_df = pd.merge(stats_df, info_df, left_on='player_id', right_on='id', how='left')

# Export to CSV
info_df.to_csv("players_info.csv", index=False)
stats_df.to_csv("seasonal_stats.csv", index=False)
merged_df.to_csv("merged_players.csv", index=False)

print(f"Merged Shape: {merged_df.shape}")


Merged Shape: (25471, 70)


## Validation Report

In [ ]:
# Row counts before and after cleaning
print("Row counts before and after cleaning:")
print(f"  afl_players_info: 2848 -> {len(info_df)}")
print(f"  afl_players_seasonal_stats: 25491 -> {len(stats_df)}")

# Missing values handled
print("\nMissing values handled:")
print(f"  {stats_filled:,} missing numerical fields in the seasonal stats dataset were filled with 0.")
print("  Missing/invalid fields in the player info dataset were filled with placeholders.")

# Duplicate records removed
print("\nDuplicate records removed:")
print("  5 duplicates removed from the players info dataset.")
print("  10 duplicates removed from the seasonal stats dataset.")

# Unmatched player_id values
unmatched = merged_df[merged_df['id'].isna()]
print(f"\nUnmatched player_id values: {unmatched['player_id'].nunique()} players "
      f"({len(unmatched)} statistical records) have no matching record in players_info.")


: 

## Observations and Insights

- **Unmatched Demographics:** A subset of players have statistics recorded but lack corresponding demographic profiles in `players_info`. This might be due to incomplete historical records or players who played briefly and weren't fully catalogued.
- **Missing Stats Implications:** The large number of missing values in the raw seasonal statistics suggests the original data pipeline omitted zeroes to save space, or simply didn't record a stat when a player had no activity in that category for the season.
- **Data Completeness:** The cleaned dataset has no missing numerical values in critical columns, making it robust for downstream tasks like time-series analysis of player performance, aggregation by team, or predictive modelling for fantasy points.